# UMAP — Uniform Manifold Approximation and Projection

LEVEL: 🟢 Concept + sklearn/library

Goal:

```text
Learn how UMAP reduces high-dimensional data into
2D / 3D while trying to preserve local structure.
```

We do NOT need to implement UMAP from scratch.  
Focus on:

1. Why UMAP?
2. Intuition
3. Internal working
4. Important mathematics
5. Important parameters
6. sklearn-style implementation
7. PCA vs t-SNE vs UMAP

## What's in this notebook

- 1\. Why UMAP?
- 2\. Intuition
- 3\. Manifold learning
- 4\. Core idea of UMAP
- 5\. Step-by-step internal working
- 6\. Important parameter — n_neighbors
- 7\. Important parameter — min_dist
- 8\. Important parameter — metric
- 9\. Important parameter — n_components
- 10\. Important parameters summary
- 15\. Basic UMAP
- 16\. Visualize UMAP
- 17\. Try different n_neighbors
- 18\. Try different min_dist
- 19\. UMAP with cosine distance
- 20\. Fit vs fit_transform
- 21\. PCA → UMAP
- 22\. UMAP for new data
- 23\. UMAP vs PCA
- 24\. UMAP vs t-SNE
- 25\. PCA vs LDA vs t-SNE vs UMAP
- 26\. Important UMAP limitations
- 27\. When should you use UMAP?
- 28\. When not to use UMAP?
- 29\. Interview questions
- 30\. 30-second interview explanation
- 31\. Working tree
- 32\. UMAP mental model

# 1. WHY UMAP?

Suppose every data point has 100 features:

X.shape = (10000, 100)

We cannot easily visualize this.

UMAP converts:

```text
100 dimensions
      ↓
   2 dimensions
```

so that we can visualize the structure of the data.

Example:

High-dimensional data

```text
Customer 1  -> 100 features
Customer 2  -> 100 features
Customer 3  -> 100 features
...
```

UMAP:

```text
100D → 2D
```

Then we can plot the points.

Main idea:

```text
Preserve important neighborhood relationships
while reducing dimensionality.
```

UMAP is especially useful for:

- Visualization
- Exploring clusters
- Understanding high-dimensional datasets
- Embedding visualization
- Preprocessing in some ML workflows

IMPORTANT:

UMAP is NOT simply "PCA but better".

PCA:

```text
preserves global linear variance
```

UMAP:

```text
tries to preserve local/manifold structure
```

t-SNE:

```text
strongly focuses on local neighborhood structure
```

# 2. INTUITION

Imagine a 3D Swiss Roll:

```text
     __________
   /          /
  /          /
 /          /
/__________/
```

The data may actually lie on a lower-dimensional surface  
inside the 3D space.

This hidden lower-dimensional structure is called a  
MANIFOLD.

UMAP tries to discover this structure.

Simple mental model:

```text
High-dimensional points
        ↓
Find nearest neighbors
        ↓
Build neighborhood graph
        ↓
Represent that graph in low dimensions
        ↓
Optimize positions
```

The important thing is:

```text
"Who is close to whom?"
```

rather than simply:

```text
"What direction contains the most variance?"
```

# 3. MANIFOLD LEARNING

UMAP belongs to:

```text
Manifold Learning
```

Other manifold-learning algorithms include:

- t-SNE
- Isomap
- Locally Linear Embedding
- Spectral Embedding
- UMAP

Basic assumption:

```text
High-dimensional data may actually lie on a
lower-dimensional manifold.
```

Example:

100-dimensional data

```text
↓
```

actual meaningful structure may effectively be  
2D or 3D.

UMAP tries to recover that structure.

# 4. CORE IDEA OF UMAP

UMAP approximately does:

```text
STEP 1
Find nearest neighbors in high-dimensional space

         ↓

STEP 2
Build a weighted neighborhood graph

         ↓

STEP 3
Initialize points in low-dimensional space

         ↓

STEP 4
Optimize the low-dimensional positions

so that nearby points remain nearby.
```

Very simplified:

```text
HIGH-DIMENSIONAL SPACE

A ---- B
 \    /
  \  /
   C

A, B and C are neighbors.


        ↓ UMAP


LOW-DIMENSIONAL SPACE

  A --- B
   \   /
     C
```

The neighborhood relationships are preserved.

# 5. STEP-BY-STEP INTERNAL WORKING

## STEP 1 — INPUT DATA

Suppose:

X =

```text
[x1, x2, x3, ..., x100]
```

for each sample.

Example:

```text
X.shape = (10000, 100)
```

UMAP receives these high-dimensional points.

## STEP 2 — FIND NEAREST NEIGHBORS

For every point:

```text
Find its nearest neighbors.
```

Example:

Point A:

```text
nearest neighbors:

B, C, D, E
```

Point B:

```text
nearest neighbors:

A, C, F
```

This creates a neighborhood graph.

```text
A -------- B
| \       /
|  \     /
C -------- D
```

The edges represent neighborhood relationships.

## STEP 3 — CONVERT DISTANCES INTO CONNECTION STRENGTH

UMAP does not simply say:

```text
neighbor = YES
neighbor = NO
```

Instead, it gives relationships a strength.

Example:

```text
A ---- B     strong relationship

A -- C       weaker relationship

A - D        very weak relationship
```

Conceptually:

```text
close points
    ↓
strong connection

far points
    ↓
weak connection
```

## STEP 4 — BUILD FUZZY NEIGHBORHOOD GRAPH

This is one of the important ideas behind UMAP.

UMAP represents local relationships using a  
fuzzy simplicial set / fuzzy graph.

You don't need to memorize the topology theory for interviews.

Interview-level understanding:

```text
UMAP builds a weighted graph describing
local neighborhood relationships.
```

## STEP 5 — INITIALIZE LOW-DIMENSIONAL POINTS

UMAP now creates:

```text
y1, y2, ..., yn
```

in low-dimensional space.

For example:

```text
100D → 2D
```

so every point becomes:

```text
(y1, y2)
```

## STEP 6 — OPTIMIZE THE LOW-DIMENSIONAL SPACE

Now UMAP tries to arrange the points so that:

```text
High-dimensional neighbors
        ↓
remain neighbors
```

If two points were strongly connected in the  
original space:

```text
pull them together
```

If two points should not be close:

```text
push them apart
```

This optimization is performed using gradient-based  
optimization.

Simplified objective:

```text
Preserve high-dimensional graph

               ↓

Match it with low-dimensional graph
```

# 6. IMPORTANT PARAMETER — n_neighbors

n_neighbors controls:

```text
"How much local neighborhood should UMAP consider?"
```

Small n_neighbors:

```text
focuses strongly on local structure.
```

Example:

```text
n_neighbors = 5

→ very local relationships
```

Large n_neighbors:

```text
considers a larger neighborhood.
```

Example:

```text
n_neighbors = 100

→ more global structure
```

Mental model:

```text
n_neighbors ↓
    ↓
more local

n_neighbors ↑
    ↓
more global
```

This is one of the MOST IMPORTANT UMAP parameters.

# 7. IMPORTANT PARAMETER — min_dist

min_dist controls how tightly points are allowed  
to pack together in the low-dimensional representation.

Small min_dist:

```text
points can form very tight clusters.
```

Large min_dist:

```text
points remain more spread out.
```

Example:

```text
min_dist = 0.0

→ very compact clusters


min_dist = 0.5

→ more spread-out representation
```

IMPORTANT:

min_dist mainly affects the compactness of clusters  
in the visualization.

# 8. IMPORTANT PARAMETER — metric

metric determines how distance between points is measured.

Common choices:

```text
"euclidean"
"manhattan"
"cosine"
"minkowski"
```

For normal numerical data:

```text
metric="euclidean"
```

For embeddings:

```text
metric="cosine"
```

is often useful because cosine similarity focuses  
on direction rather than magnitude.

# 9. IMPORTANT PARAMETER — n_components

Number of output dimensions.

Example:

```text
n_components=2

100D → 2D


n_components=3

100D → 3D
```

For visualization:

```text
usually 2 or 3.
```

# 10. IMPORTANT PARAMETERS SUMMARY

n_neighbors:

```text
Controls local vs broader structure.

Small → more local
Large → more global
```

min_dist:

```text
Controls compactness of points.

Small → tighter clusters
Large → more spread
```

metric:

```text
Distance function.
```

n_components:

```text
Output dimensions.
```

random_state:

```text
Makes results more reproducible.
```

spread:

```text
Controls overall effective scale of embedded points.
```

learning_rate:

```text
Controls optimization step size.
```

# 11. INSTALLATION

If UMAP is not installed:

pip install umap-learn

In Jupyter:

%pip install umap-learn

# 12. IMPORTS

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

import umap.umap_ as umap

# 13. LOAD DATA

Digits dataset:

Each image is represented by many pixel features.

We will reduce it to 2 dimensions.

In [ ]:
digits = load_digits()

X = digits.data
y = digits.target

print("Original shape:", X.shape)

# 14. SCALE DATA

Scaling can be important because UMAP uses distances.

If one feature has a much larger scale than another,  
it can dominate distance calculations.

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Scaled shape:", X_scaled.shape)

# 15. BASIC UMAP

In [ ]:
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=42
)

X_umap = reducer.fit_transform(X_scaled)

print("UMAP shape:", X_umap.shape)

# 16. VISUALIZE UMAP

In [ ]:
plt.figure(figsize=(10, 7))

plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=y,
    s=10
)

plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("UMAP Projection")

plt.show()

# 17. TRY DIFFERENT n_neighbors

Small n_neighbors:

```text
focuses more on local structure.
```

Large n_neighbors:

```text
considers broader structure.
```

In [ ]:
for neighbors in [5, 15, 50]:

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=neighbors,
        min_dist=0.1,
        random_state=42
    )

    embedding = reducer.fit_transform(X_scaled)

    plt.figure(figsize=(8, 6))

    plt.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=y,
        s=10
    )

    plt.title(f"UMAP — n_neighbors={neighbors}")
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")

    plt.show()

# 18. TRY DIFFERENT min_dist

Smaller min_dist:

```text
tighter clusters
```

Larger min_dist:

```text
more spread-out clusters
```

In [ ]:
for min_dist in [0.0, 0.1, 0.5]:

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=min_dist,
        random_state=42
    )

    embedding = reducer.fit_transform(X_scaled)

    plt.figure(figsize=(8, 6))

    plt.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=y,
        s=10
    )

    plt.title(f"UMAP — min_dist={min_dist}")
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")

    plt.show()

# 19. UMAP WITH COSINE DISTANCE

Cosine distance is especially useful for some  
high-dimensional embeddings.

Example:

```text
sentence embeddings
document embeddings
semantic vectors
```

In [ ]:
reducer_cosine = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

X_umap_cosine = reducer_cosine.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))

plt.scatter(
    X_umap_cosine[:, 0],
    X_umap_cosine[:, 1],
    c=y,
    s=10
)

plt.title("UMAP with Cosine Distance")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")

plt.show()

# 20. FIT vs FIT_TRANSFORM

Similar to sklearn transformers:

fit(X):

```text
learns the UMAP structure.
```

fit_transform(X):

```text
learns the structure AND transforms the data.
```

Example:

reducer.fit(X_scaled)

X_umap = reducer.transform(X_scaled)

This is useful when you want to reuse the fitted reducer.

# 21. PCA → UMAP

For very high-dimensional data, a common workflow is:

```text
Original data
     ↓
    PCA
     ↓
moderate dimensions
     ↓
   UMAP
     ↓
   2D/3D
```

Example:

```text
1000D
  ↓
PCA → 50D
  ↓
UMAP → 2D
```

This can make UMAP faster and reduce noise.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=30)

X_pca = pca.fit_transform(X_scaled)

print("After PCA:", X_pca.shape)

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)

X_pca_umap = reducer.fit_transform(X_pca)

plt.figure(figsize=(10, 7))

plt.scatter(
    X_pca_umap[:, 0],
    X_pca_umap[:, 1],
    c=y,
    s=10
)

plt.title("PCA → UMAP")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")

plt.show()

# 22. UMAP FOR NEW DATA

After fitting UMAP:

```text
reducer.fit(X_train)
```

we can transform new data:

```text
reducer.transform(X_test)
```

Example:

In [ ]:
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)

reducer.fit(X_scaled)

X_train_umap = reducer.transform(X_scaled)

print(X_train_umap.shape)

# 23. UMAP vs PCA

PCA:

```text
Linear dimensionality reduction

Finds directions of maximum variance.
```

UMAP:

```text
Non-linear dimensionality reduction

Tries to preserve neighborhood/manifold structure.
```

PCA:

```text
Fast
Stable
Easier to interpret
```

UMAP:

```text
Can capture non-linear structure
Excellent for visualization
More hyperparameters
```

Mental model:

PCA:

```text
"Find the best straight directions."
```

UMAP:

```text
"Preserve who is close to whom."
```

# 24. UMAP vs t-SNE

Both are non-linear dimensionality reduction techniques.

t-SNE:

```text
Mainly focused on local neighborhoods.

Excellent visualization.

Computationally expensive for large datasets.
```

UMAP:

```text
Also preserves local structure.

Usually faster.

Often gives better preservation of broader structure.

Can transform new points after fitting.
```

Important:

Neither should automatically be treated as a  
"better clustering algorithm".

They are primarily dimensionality-reduction / visualization  
techniques.

# 25. PCA vs LDA vs t-SNE vs UMAP

### PCA

Supervised?       NO  
Linear?           YES  
Main goal?        Maximize variance  
Labels required?  NO

### LDA

Supervised?       YES  
Linear?           YES  
Main goal?        Maximize class separation  
Labels required?  YES

### t-SNE

Supervised?       NO  
Linear?           NO  
Main goal?        Preserve local neighborhoods  
Main use?         Visualization

### UMAP

Supervised?       NO (standard UMAP)  
Linear?           NO  
Main goal?        Preserve manifold/neighborhood structure  
Main use?         Visualization / dimensionality reduction

# 26. IMPORTANT UMAP LIMITATIONS

1. It is stochastic.

Different runs can produce somewhat different embeddings  
unless random_state is controlled.

2. Distances in the 2D plot should not automatically be  
interpreted as exact original-space distances.

3. Cluster sizes in the visualization can be misleading.

4. Hyperparameters can significantly change the visualization.

5. Scaling and metric choice matter.

6. A visually separated cluster does NOT automatically mean  
that the original data contains a true natural cluster.

7. UMAP should not automatically be used as a preprocessing  
step for every ML model.

# 27. WHEN SHOULD YOU USE UMAP?

USE UMAP WHEN:

- You have high-dimensional data
- You want visualization
- You want to explore local structure
- You want to inspect embeddings
- Data has possible non-linear structure

Examples:

```text
Image embeddings
NLP embeddings
Gene-expression data
Customer embeddings
Recommendation embeddings
Deep-learning representations
```

# 28. WHEN NOT TO USE UMAP?

Avoid blindly using UMAP when:

- You need interpretable linear components
- You need exact global distances
- You need a deterministic transformation
- You don't have enough data for meaningful neighborhoods
- You only need a simple dimensionality reduction

In those cases:

```text
PCA may be simpler.
```

# 29. INTERVIEW QUESTIONS

Q1. What is UMAP?

Answer:

UMAP is a non-linear dimensionality-reduction technique  
based on manifold learning. It builds a neighborhood graph  
in high-dimensional space and optimizes a low-dimensional  
representation that preserves important neighborhood  
relationships.

Q2. What does n_neighbors control?

Answer:

It controls the size of the local neighborhood used by UMAP.  
Smaller values emphasize local structure, while larger values  
capture broader structure.

Q3. What does min_dist control?

Answer:

It controls how tightly points are allowed to cluster in the  
low-dimensional representation.

Q4. UMAP vs PCA?

Answer:

PCA is linear and maximizes variance.  
UMAP is non-linear and focuses on preserving  
neighborhood/manifold structure.

Q5. UMAP vs t-SNE?

Answer:

Both are non-linear visualization techniques focused on  
neighborhood structure. UMAP generally provides a faster  
and more flexible embedding and supports transforming new  
points after fitting.

Q6. Is UMAP supervised?

Standard UMAP is unsupervised.

However, UMAP also supports supervised/semi-supervised  
variants where labels can influence the embedding.

Q7. Why scale data before UMAP?

Because UMAP uses distances, and features with very large  
scales can dominate those distances.

# 30. 30-SECOND INTERVIEW EXPLANATION

"UMAP is a non-linear dimensionality reduction technique  
based on manifold learning. It first finds nearest neighbors  
of each point in the high-dimensional space and builds a  
weighted neighborhood graph. It then optimizes a low-dimensional  
representation so that important neighborhood relationships  
are preserved. The two important parameters are n_neighbors,  
which controls local versus broader structure, and min_dist,  
which controls how tightly points can be packed in the  
resulting embedding."

# 31. WORKING TREE

```text
      HIGH-DIMENSIONAL DATA
               |
               v
      Find nearest neighbors
               |
               v
    Build weighted neighborhood
             graph
               |
               v
    Fuzzy neighborhood graph
               |
               v
Initialize low-dimensional points
               |
               v
   Optimize low-dimensional
       representation
         /           \
        /             \
 Pull neighbors     Push apart
        \             /
         \           /
          \         /
           v       v
      Final embedding
           |
           v
         2D / 3D
```

# 32. UMAP MENTAL MODEL

PCA:

```text
"Which directions explain the most variance?"
```

LDA:

```text
"Which directions separate my classes?"
```

t-SNE:

```text
"Which points should remain local neighbors?"
```

UMAP:

```text
"What is the neighborhood/manifold structure,
 and how can I preserve it in fewer dimensions?"
```

# END OF UMAP